# Collected Data Evaluation — Two-Stage Pipeline

Evaluates the fine-tuned M3 two-stage system on your own collected CSV data.

## Pipeline
```
Raw CSV files (ax_g, ay_g, az_g, gx_dps, gy_dps, gz_dps)
        │
        ▼
  Label from filename  (F** = fall, D** = ADL)
        │
        ▼
  Windowing — 50 samples, stride 20  (same as SisFall preprocessing)
  Falls  → impact peak detection → 1 centred 50-sample window
  ADLs   → hard negatives (top-3) + easy negatives (random 5)
        │
        ▼
  [GATE MODEL]  — StandardScaler + LogReg (3 active features)
  Threshold: 0.58
        │
   ┌────┴────┐
  ADL      Potential Fall
 reject         │
                ▼
          [M3 CNN]  model_dense64.tflite
                │
                ▼
          Final prediction
```

## Key preprocessing decisions (matched to SisFall training)
- No unit conversion needed — your CSVs are already in g and dps
- No normalization — raw physical units, same as training
- Window: 50 samples = 500ms at 100 Hz
- Stride: 20 samples (60% overlap)
- Fall labeling: one centred impact window per trial (clean verifier)
- ADL labeling: hard negatives (acc ≥ HARD_THRESH, top-3) + easy negatives (acc < EASY_THRESH, random 5)

## 1. Setup & Imports

In [2]:
import numpy as np
import pandas as pd
import os
import glob
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

print(f'NumPy     : {np.__version__}')
print(f'TensorFlow: {tf.__version__}')
print(f'Pandas    : {pd.__version__}')

NumPy     : 2.0.2
TensorFlow: 2.20.0
Pandas    : 2.2.2


## 2. Configuration — Edit Paths Here

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── EDIT THESE ────────────────────────────────────────────────────────
CSV_DIR     = '/content/drive/MyDrive/M4/arduino_dataset/Data Collection'   # folder containing your CSV files
TFLITE_PATH = '/content/drive/MyDrive/M3 Project/baseline_model/final_cnn_v2.tflite'                    # put alongside this notebook
# ─────────────────────────────────────────────────────────────────────

# Windowing — matched to SisFall preprocessing
WIN_SAMPLES  = 50    # 500ms at 100 Hz
STRIDE       = 20    # 60% overlap
FALL_PRE     = 25    # samples before impact peak
FALL_POST    = 25    # samples after impact peak

# ADL mining thresholds — from analysis_output.json (Notebook 2)
HARD_THRESH  = 1.569   # 75th percentile of ADL window peaks
EASY_THRESH  = 1.072   # 25th percentile of ADL window peaks
MAX_HARD     = 3
MAX_EASY     = 5

# Impact detection thresholds — from analysis_output.json
STILL_MEAN     = 1.25
MIN_STILL      = 150
POST_STILL_MAX = 1.6

# Gate
GATE_THRESHOLD = 0.58

# Random seed
RANDOM_SEED = 42
rng = np.random.RandomState(RANDOM_SEED)

assert os.path.exists(CSV_DIR),     f'CSV folder not found: {CSV_DIR}'
assert os.path.exists(TFLITE_PATH), f'TFLite model not found: {TFLITE_PATH}'
print('Paths OK')

Paths OK


In [ ]:
import os

data_collection_path = "/content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama"

EXPECTED_HEADER = "t_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps"

cleaned = 0
errors = []

for root, dirs, files in os.walk(data_collection_path):
    for filename in files:
        if filename.endswith(".csv"):
            filepath = os.path.join(root, filename)
            try:
                with open(filepath, "r") as f:
                    lines = f.readlines()

                # Find the header line
                header_index = None
                for i, line in enumerate(lines):
                    if line.strip() == EXPECTED_HEADER:
                        header_index = i
                        break

                if header_index is None:
                    errors.append(f"⚠️ Header not found: {filepath}")
                    continue

                # Keep from header to second-to-last line (remove last line)
                cleaned_lines = lines[header_index:-1]

                with open(filepath, "w") as f:
                    f.writelines(cleaned_lines)

                print(f"✅ Cleaned: {filepath}")
                cleaned += 1

            except Exception as e:
                errors.append(f"❌ Error: {filepath} → {e}")

print(f"\n✅ Done! {cleaned} file(s) cleaned.")
if errors:
    for err in errors:
        print(err)

✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D01_01.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D01_02.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D01_03.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D01_04.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D01_05.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D02_01.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D02_02.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D02_03.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D02_04.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D02_05.csv
✅ Cleaned: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D03_01.csv
✅ Cleaned: /content/drive/MyDriv

## 3. Load CSV Files

Reads all CSVs from `CSV_DIR`. Label is parsed from the filename:
- Filename starts with `F` (e.g. `F03_01.csv`) → **fall (label = 1)**
- Filename starts with `D` (e.g. `D01_01.csv`) → **ADL (label = 0)**

In [5]:
def load_csv(filepath):
    """
    Load a single trial CSV from the Arduino data collection firmware.

    File structure:
      - File may begin with a leftover STREAM END block (previous session artifact)
      - Real data is between === STREAM BEGIN === and the NEXT === STREAM END ===
      - Header row: t_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps
      - Data rows follow until === STREAM END ===

    Returns: accel (N,3) in g, gyro (N,3) in dps
    Returns (None, None) if fewer than WIN_SAMPLES rows parsed.
    """
    rows = []
    in_stream = False   # True only AFTER === STREAM BEGIN === is seen
    header_seen = False

    with open(filepath, "rb") as f:
        for line in f:
            line = line.decode("latin-1").strip()

            # Start collecting only after STREAM BEGIN
            if "=== STREAM BEGIN ===" in line:
                in_stream = True
                header_seen = False
                rows = []   # reset in case of multiple streams in one file
                continue

            # Stop at STREAM END — but only if we already entered a stream
            if "=== STREAM END ===" in line and in_stream:
                break

            if not in_stream:
                continue

            # Skip header row
            if not header_seen:
                if line.startswith("t_ms"):
                    header_seen = True
                continue

            # Skip blank, comment, or non-CSV lines (e.g. stray "s")
            if not line or line.startswith("#") or "," not in line:
                continue

            parts = line.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                continue

    if len(rows) < WIN_SAMPLES:
        return None, None

    arr   = np.array(rows, dtype=np.float32)
    # Columns: t_ms(0), ax_g(1), ay_g(2), az_g(3), gx_dps(4), gy_dps(5), gz_dps(6)
    accel = arr[:, 1:4]   # ax, ay, az already in g
    gyro  = arr[:, 4:7]   # gx, gy, gz already in dps
    return accel, gyro
def smv(a):
    """Signal Magnitude Vector: sqrt(x^2 + y^2 + z^2) per sample."""
    return np.sqrt(np.sum(a ** 2, axis=1))


csv_files = sorted(glob.glob(os.path.join(CSV_DIR, '**', '*.csv'), recursive=True))
print(f'Found {len(csv_files)} CSV files in {CSV_DIR}')

fall_trials = []   # list of dicts
adl_trials  = []   # list of dicts
skipped     = []

for filepath in csv_files:
    fname = os.path.basename(filepath)
    prefix = fname[0].upper()

    if prefix == 'F':
        label = 1
    elif prefix == 'D':
        label = 0
    else:
        skipped.append(fname)
        continue

    accel, gyro = load_csv(filepath)

    if accel is None:
        skipped.append(fname)
        print(f'  Warning: could not parse {fname}')
        continue

    if len(accel) < WIN_SAMPLES:
        skipped.append(fname)
        continue

    trial = {
        'filename': fname,
        'label':    label,
        'accel':    accel,
        'gyro':     gyro,
        'a_smv':    smv(accel),
        'g_smv':    smv(gyro),
    }

    if label == 1:
        fall_trials.append(trial)
    else:
        adl_trials.append(trial)

print(f'\nLoaded:')
print(f'  Fall trials : {len(fall_trials)}')
print(f'  ADL trials  : {len(adl_trials)}')
if skipped:
    print(f'  Skipped     : {skipped}')

Found 374 CSV files in /content/drive/MyDrive/M4/arduino_dataset/Data Collection


KeyboardInterrupt: 

In [7]:
# Check what a cleaned file actually looks like now
sample_file = "/content/drive/MyDrive/M4/arduino_dataset/Data Collection/V1_Sama/D01_01.csv"

with open(sample_file, "rb") as f:
    for i, line in enumerate(f):
        print(repr(line.decode("latin-1").strip()))
        if i >= 10:
            break

't_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps'
'0,-0.1015,0.9616,-0.0185,-1.680,1.120,-0.840'
'10,-0.1083,0.9604,-0.0168,-0.910,1.120,-0.910'
'20,-0.1108,0.9606,-0.0137,-0.630,1.050,-0.770'
'30,-0.1108,0.9606,-0.0137,-0.630,1.050,-0.770'
'40,-0.1135,0.9609,-0.0129,-0.420,1.260,-0.630'
'50,-0.1135,0.9609,-0.0129,-0.420,1.260,-0.630'
'60,-0.1154,0.9597,-0.0095,-0.490,1.330,-0.420'
'70,-0.1147,0.9626,-0.0105,-0.770,1.540,-0.140'
'80,-0.1147,0.9626,-0.0105,-0.770,1.540,-0.140'
'90,-0.1171,0.9628,-0.0100,-1.190,1.960,0.420'


In [8]:
def load_csv(filepath):
    """
    Load a cleaned trial CSV.
    File structure after cleaning:
      - First line: header (t_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps)
      - Remaining lines: data rows

    Returns: accel (N,3) in g, gyro (N,3) in dps
    Returns (None, None) if fewer than WIN_SAMPLES rows parsed.
    """
    rows = []

    with open(filepath, "rb") as f:
        for i, line in enumerate(f):
            line = line.decode("latin-1").strip()

            # Skip header
            if i == 0:
                if not line.startswith("t_ms"):
                    return None, None  # Unexpected format
                continue

            # Skip blank, comment, or non-CSV lines
            if not line or line.startswith("#") or "," not in line:
                continue

            parts = line.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                continue

    if len(rows) < WIN_SAMPLES:
        return None, None

    arr   = np.array(rows, dtype=np.float32)
    accel = arr[:, 1:4]   # ax, ay, az in g
    gyro  = arr[:, 4:7]   # gx, gy, gz in dps
    return accel, gyro

In [9]:
filepath = '/content/drive/MyDrive/M3 Project/collected data/session1/D01_01.csv'

with open(filepath, 'rb') as f:
    for i, line in enumerate(f):
        print(i, repr(line[:80]))
        if i > 15:
            break

0 b's=== STREAM BEGIN ===\r\n'
1 b't_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps\r\n'
2 b'0,-0.1015,0.9616,-0.0185,-1.680,1.120,-0.840\r\n'
3 b'10,-0.1083,0.9604,-0.0168,-0.910,1.120,-0.910\r\n'
4 b'20,-0.1108,0.9606,-0.0137,-0.630,1.050,-0.770\r\n'
5 b'30,-0.1108,0.9606,-0.0137,-0.630,1.050,-0.770\r\n'
6 b'40,-0.1135,0.9609,-0.0129,-0.420,1.260,-0.630\r\n'
7 b'50,-0.1135,0.9609,-0.0129,-0.420,1.260,-0.630\r\n'
8 b'60,-0.1154,0.9597,-0.0095,-0.490,1.330,-0.420\r\n'
9 b'70,-0.1147,0.9626,-0.0105,-0.770,1.540,-0.140\r\n'
10 b'80,-0.1147,0.9626,-0.0105,-0.770,1.540,-0.140\r\n'
11 b'90,-0.1171,0.9628,-0.0100,-1.190,1.960,0.420\r\n'
12 b'100,-0.1171,0.9628,-0.0100,-1.190,1.960,0.910\r\n'
13 b'110,-0.1183,0.9636,-0.0105,-1.260,2.520,0.910\r\n'
14 b'120,-0.1169,0.9658,-0.0129,-1.260,2.450,1.260\r\n'
15 b'130,-0.1169,0.9658,-0.0129,-1.260,2.450,1.260\r\n'
16 b'140,-0.1159,0.9670,-0.0171,-0.980,2.030,1.400\r\n'


In [10]:
def load_csv(filepath):
    """
    Load a cleaned trial CSV.
    File structure after cleaning:
      - First line: header (t_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps)
      - Remaining lines: data rows

    Returns: accel (N,3) in g, gyro (N,3) in dps
    Returns (None, None) if fewer than WIN_SAMPLES rows parsed.
    """
    rows = []

    with open(filepath, "rb") as f:
        for i, line in enumerate(f):
            line = line.decode("latin-1").strip()

            # Skip header
            if i == 0:
                if not line.startswith("t_ms"):
                    return None, None  # Unexpected format
                continue

            # Skip blank, comment, or non-CSV lines
            if not line or line.startswith("#") or "," not in line:
                continue

            parts = line.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                continue

    if len(rows) < WIN_SAMPLES:
        return None, None

    arr   = np.array(rows, dtype=np.float32)
    accel = arr[:, 1:4]   # ax, ay, az in g
    gyro  = arr[:, 4:7]   # gx, gy, gz in dps
    return accel, gyro

In [11]:
import os

# Find any csv file automatically
import glob

csv_files = sorted(glob.glob("/content/drive/MyDrive/M4/arduino_dataset/Data Collection/**/*.csv", recursive=True))

print(f"Total CSV files found: {len(csv_files)}")
print(f"\nFirst file: {csv_files[0]}\n")

# Print first 15 lines
with open(csv_files[0], "rb") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {repr(line.decode('latin-1').strip())}")
        if i >= 15:
            break

Total CSV files found: 374

First file: /content/drive/MyDrive/M4/arduino_dataset/Data Collection/V10_Isham/D01_01.csv

Line 0: 't_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps'
Line 1: '0,-0.0127,0.9994,-0.0063,1.050,2.380,-0.560'
Line 2: '10,-0.0093,1.0021,-0.0068,0.700,2.310,-0.700'
Line 3: '20,-0.0063,1.0063,-0.0061,0.630,2.170,-0.700'
Line 4: '30,-0.0027,1.0063,-0.0044,0.700,2.030,-0.770'
Line 5: '40,-0.0029,1.0063,-0.0044,0.980,1.820,-0.770'
Line 6: '50,0.0000,1.0043,-0.0051,1.050,1.890,-0.770'
Line 7: '60,0.0002,1.0031,-0.0049,1.400,1.820,-0.700'
Line 8: '70,-0.0005,1.0006,-0.0049,2.100,1.540,-0.700'
Line 9: '80,-0.0044,0.9963,-0.0078,1.820,1.120,-0.560'
Line 10: '90,-0.0063,0.9950,-0.0078,0.840,0.630,-0.560'
Line 11: '99,-0.0063,0.9963,-0.0085,0.350,0.560,-0.490'
Line 12: '109,-0.0056,0.9994,-0.0090,0.140,0.630,-0.420'
Line 13: '119,-0.0039,1.0006,-0.0102,-0.070,0.420,-0.350'
Line 14: '129,-0.0046,1.0028,-0.0115,0.000,0.140,-0.350'
Line 15: '139,-0.0034,1.0041,-0.0122,0.000,0.000,-0.1

In [12]:
# Diagnose exactly why files are being skipped
for filepath in csv_files[:5]:
    with open(filepath, "rb") as f:
        lines = f.readlines()

    data_rows = len(lines) - 1  # subtract header
    print(f"\n{os.path.basename(filepath)}:")
    print(f"  Total rows (excl. header): {data_rows}")
    print(f"  WIN_SAMPLES: {WIN_SAMPLES}")
    print(f"  Passes WIN_SAMPLES check: {data_rows >= WIN_SAMPLES}")

    # Try parsing manually
    parsed = 0
    for line in lines[1:]:
        line = line.decode("latin-1").strip()
        if not line or "," not in line:
            continue
        parts = line.split(",")
        if len(parts) != 7:
            print(f"  ⚠️ Bad line (not 7 cols): {repr(line)}")
            continue
        try:
            [float(v) for v in parts]
            parsed += 1
        except ValueError as e:
            print(f"  ⚠️ ValueError: {repr(line)} → {e}")

    print(f"  Successfully parsed rows: {parsed}")
    print(f"  Passes after parsing: {parsed >= WIN_SAMPLES}")


D01_01.csv:
  Total rows (excl. header): 10000
  WIN_SAMPLES: 50
  Passes WIN_SAMPLES check: True
  Successfully parsed rows: 10000
  Passes after parsing: True

D01_02.csv:
  Total rows (excl. header): 10000
  WIN_SAMPLES: 50
  Passes WIN_SAMPLES check: True
  Successfully parsed rows: 10000
  Passes after parsing: True

D01_03.csv:
  Total rows (excl. header): 10000
  WIN_SAMPLES: 50
  Passes WIN_SAMPLES check: True
  Successfully parsed rows: 10000
  Passes after parsing: True

D01_04.csv:
  Total rows (excl. header): 10000
  WIN_SAMPLES: 50
  Passes WIN_SAMPLES check: True
  Successfully parsed rows: 10000
  Passes after parsing: True

D01_05.csv:
  Total rows (excl. header): 10000
  WIN_SAMPLES: 50
  Passes WIN_SAMPLES check: True
  Successfully parsed rows: 10000
  Passes after parsing: True


In [13]:
import inspect
print(inspect.getsource(load_csv))

def load_csv(filepath):
    """
    Load a cleaned trial CSV.
    File structure after cleaning:
      - First line: header (t_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps)
      - Remaining lines: data rows

    Returns: accel (N,3) in g, gyro (N,3) in dps
    Returns (None, None) if fewer than WIN_SAMPLES rows parsed.
    """
    rows = []

    with open(filepath, "rb") as f:
        for i, line in enumerate(f):
            line = line.decode("latin-1").strip()

            # Skip header
            if i == 0:
                if not line.startswith("t_ms"):
                    return None, None  # Unexpected format
                continue

            # Skip blank, comment, or non-CSV lines
            if not line or line.startswith("#") or "," not in line:
                continue

            parts = line.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
             

In [14]:
def load_csv(filepath):
    rows = []

    with open(filepath, "rb") as f:
        for i, line in enumerate(f):
            line = line.decode("latin-1").strip()

            if i == 0:
                if not line.startswith("t_ms"):
                    return None, None
                continue

            if not line or line.startswith("#") or "," not in line:
                continue

            parts = line.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                continue

    if len(rows) < WIN_SAMPLES:
        return None, None

    arr   = np.array(rows, dtype=np.float32)
    accel = arr[:, 1:4]
    gyro  = arr[:, 4:7]
    return accel, gyro

In [15]:
# Test load_csv directly on one file
test_file = csv_files[0]
print(f"Testing: {os.path.basename(test_file)}")

accel, gyro = load_csv(test_file)

if accel is None:
    print("❌ Returned None — checking why...")

    rows = []
    with open(test_file, "rb") as f:
        for i, line in enumerate(f):
            decoded = line.decode("latin-1").strip()
            if i == 0:
                print(f"Header line: {repr(decoded)}")
                print(f"Starts with 't_ms': {decoded.startswith('t_ms')}")
                continue
            if not decoded or decoded.startswith("#") or "," not in decoded:
                continue
            parts = decoded.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                continue

    print(f"Rows parsed manually: {len(rows)}")
    print(f"WIN_SAMPLES: {WIN_SAMPLES}")
else:
    print(f"✅ Success! accel shape: {accel.shape}, gyro shape: {gyro.shape}")

Testing: D01_01.csv
✅ Success! accel shape: (10000, 3), gyro shape: (10000, 3)


In [16]:
fall_trials = []
adl_trials  = []
skipped     = []

for filepath in csv_files:
    fname = os.path.basename(filepath)
    prefix = fname[0].upper()

    if prefix == 'F':
        label = 1
    elif prefix == 'D':
        label = 0
    else:
        skipped.append(fname)
        continue

    accel, gyro = load_csv(filepath)

    if accel is None:
        skipped.append(fname)
        print(f'  Warning: could not parse {fname}')
        continue

    trial = {
        'filename': fname,
        'label':    label,
        'accel':    accel,
        'gyro':     gyro,
        'a_smv':    smv(accel),
        'g_smv':    smv(gyro),
    }

    if label == 1:
        fall_trials.append(trial)
    else:
        adl_trials.append(trial)

print(f'\nLoaded:')
print(f'  Fall trials : {len(fall_trials)}')
print(f'  ADL trials  : {len(adl_trials)}')
if skipped:
    print(f'  Skipped     : {skipped}')


Loaded:
  Fall trials : 147
  ADL trials  : 222
  Skipped     : ['D11_01.csv', 'F04_05.csv', 'D07_03.csv', 'D17_01.csv', 'D02_01.csv']


In [ ]:
import glob
import os

csv_files = sorted(glob.glob("/content/drive/MyDrive/M4/arduino_dataset/Data Collection/**/*.csv", recursive=True))

fall_count = 0
adl_count  = 0
other      = []

for filepath in csv_files:
    fname = os.path.basename(filepath)
    prefix = fname[0].upper()
    if prefix == 'F':
        fall_count += 1
    elif prefix == 'D':
        adl_count += 1
    else:
        other.append(fname)

print(f"Total CSV files : {len(csv_files)}")
print(f"Fall trials (F*): {fall_count}")
print(f"ADL  trials (D*): {adl_count}")
if other:
    print(f"Other (skipped) : {other}")

Total CSV files : 374
Fall trials (F*): 148
ADL  trials (D*): 226


## 4. Windowing

### Falls → Impact peak detection → 1 centred 50-sample window
Same clean verifier strategy as SisFall preprocessing (Notebook 6).

### ADLs → Hard negatives (top-3) + Easy negatives (random 5)
Same strategy as Experiment 4 (final model training).

In [17]:
def detect_impact(a_smv, g_smv):
    """
    Locate fall impact peak using the same 4-condition verifier as SisFall preprocessing.
    Returns sample index of impact peak, or None if not detected.

    Note: for collected data we use a simpler generic threshold since we don't
    have per-type thresholds for your custom fall types.
    Adjust ACC_THRESH / GYR_THRESH below if some falls are being missed.
    """
    ACC_THRESH = 1.5   # minimum acc SMV peak to count as fall impact
    GYR_THRESH = 90.0  # minimum gyro SMV peak within ±1s of impact
    HIGH_ACC   = 4.0   # override: if acc >= this, accept regardless of gyro

    n  = len(a_smv)
    pk = int(np.argmax(a_smv))
    ap = float(a_smv[pk])

    # Condition 1+2: acc and gyro thresholds
    gs = max(0, pk - 100); ge = min(n, pk + 100)
    gp = float(g_smv[gs:ge].max())
    if not (ap >= ACC_THRESH and (gp >= GYR_THRESH or ap >= HIGH_ACC)):
        return None

    # Condition 3: post-impact stillness
    ss = -1
    for i in range(pk + 1, n - MIN_STILL):
        r = a_smv[i:i + MIN_STILL]
        if r.mean() < STILL_MEAN:
            ss = i; break
    if ss == -1:
        return None

    # Condition 4: no resumed activity after stillness
    ch = min(ss + MIN_STILL, n)
    if ch < n and float(a_smv[ch:].max()) > POST_STILL_MAX:
        return None

    return pk


def get_fall_window(trial):
    """
    Returns the single 50-sample window centred on the detected impact peak.
    Returns None if no impact detected (trial discarded).
    """
    pk = detect_impact(trial['a_smv'], trial['g_smv'])
    if pk is None:
        return None

    raw6 = np.concatenate([trial['accel'], trial['gyro']], axis=1)
    s = pk - FALL_PRE; e = pk + FALL_POST
    if s < 0 or e > len(raw6):
        return None

    return raw6[s:e].astype(np.float32)


def get_adl_windows(trial):
    """
    Hard negatives (acc >= HARD_THRESH, top-3) +
    Easy negatives (acc < EASY_THRESH, random 5).
    Same strategy as Experiment 4 training.
    """
    raw6 = np.concatenate([trial['accel'], trial['gyro']], axis=1)
    a    = trial['a_smv']
    hard = []; easy = []

    for w in range(0, len(raw6) - WIN_SAMPLES + 1, STRIDE):
        pk  = float(a[w:w + WIN_SAMPLES].max())
        win = raw6[w:w + WIN_SAMPLES]
        if pk >= HARD_THRESH:
            hard.append((pk, win))
        elif pk < EASY_THRESH:
            easy.append(win)

    hard.sort(key=lambda x: -x[0])
    hw = [win for _, win in hard[:MAX_HARD]]

    if len(easy) > MAX_EASY:
        idx  = rng.choice(len(easy), MAX_EASY, replace=False)
        easy = [easy[i] for i in idx]

    return hw + easy


# ── Build windows ────────────────────────────────────────────────────
windows = []; labels = []
fall_detected = 0; fall_skipped = 0
fall_skip_names = []

for trial in fall_trials:
    win = get_fall_window(trial)
    if win is not None:
        windows.append(win); labels.append(1)
        fall_detected += 1
    else:
        fall_skipped += 1
        fall_skip_names.append(trial['filename'])

for trial in adl_trials:
    for win in get_adl_windows(trial):
        windows.append(win); labels.append(0)

X = np.array(windows, dtype=np.float32)
y = np.array(labels,  dtype=np.int32)

print('=' * 55)
print('WINDOWING COMPLETE')
print('=' * 55)
print(f'Total windows  : {len(y):,}')
print(f'Fall windows   : {int(y.sum())}  (from {fall_detected}/{fall_detected+fall_skipped} trials detected)')
print(f'ADL windows    : {int((y==0).sum())}')
print(f'Fall %         : {100*y.mean():.1f}%')

if fall_skipped > 0:
    print(f'\n⚠ {fall_skipped} fall trial(s) had no impact detected — discarded:')
    for name in fall_skip_names:
        print(f'  {name}')
    print('  → If this is unexpected, lower ACC_THRESH in detect_impact() above.')

WINDOWING COMPLETE
Total windows  : 1,502
Fall windows   : 140  (from 140/147 trials detected)
ADL windows    : 1362
Fall %         : 9.3%

⚠ 7 fall trial(s) had no impact detected — discarded:
  F01_02.csv
  F04_01.csv
  F01_01.csv
  F02_01.csv
  F07_01.csv
  F14_01.csv
  F02_01.csv
  → If this is unexpected, lower ACC_THRESH in detect_impact() above.


In [18]:
for trial in fall_trials:
    win       = get_fall_window(trial)
    volunteer = os.path.basename(os.path.dirname(trial['filepath']))
    if win is None:
        print(f'  [{volunteer}] {trial["filename"]}')

KeyError: 'filepath'

In [19]:
import os
import glob
import numpy as np
from collections import defaultdict

# ── Constants ────────────────────────────────────────────────────────
CSV_DIR     = "/content/drive/MyDrive/M4/arduino_dataset/Data Collection"
WIN_SAMPLES = 50
STRIDE      = 20
FALL_PRE    = 25
FALL_POST   = 25
MIN_STILL   = 50
STILL_MEAN  = 1.2
POST_STILL_MAX = 1.5
HARD_THRESH = 1.3
EASY_THRESH = 1.1
MAX_HARD    = 3
MAX_EASY    = 5
rng         = np.random.default_rng(42)

# ── Helper functions ─────────────────────────────────────────────────
def smv(a):
    return np.sqrt(np.sum(a ** 2, axis=1))

def load_csv(filepath):
    rows = []
    with open(filepath, "rb") as f:
        for i, line in enumerate(f):
            line = line.decode("latin-1").strip()
            if i == 0:
                if not line.startswith("t_ms"):
                    return None, None
                continue
            if not line or line.startswith("#") or "," not in line:
                continue
            parts = line.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                continue
    if len(rows) < WIN_SAMPLES:
        return None, None
    arr   = np.array(rows, dtype=np.float32)
    accel = arr[:, 1:4]
    gyro  = arr[:, 4:7]
    return accel, gyro

def detect_impact(a_smv, g_smv):
    ACC_THRESH = 1.5
    GYR_THRESH = 90.0
    HIGH_ACC   = 4.0

    n  = len(a_smv)
    pk = int(np.argmax(a_smv))
    ap = float(a_smv[pk])

    gs = max(0, pk - 100); ge = min(n, pk + 100)
    gp = float(g_smv[gs:ge].max())
    if not (ap >= ACC_THRESH and (gp >= GYR_THRESH or ap >= HIGH_ACC)):
        return None

    ss = -1
    for i in range(pk + 1, n - MIN_STILL):
        r = a_smv[i:i + MIN_STILL]
        if r.mean() < STILL_MEAN:
            ss = i; break
    if ss == -1:
        return None

    ch = min(ss + MIN_STILL, n)
    if ch < n and float(a_smv[ch:].max()) > POST_STILL_MAX:
        return None

    return pk

def get_fall_window(trial):
    pk = detect_impact(trial['a_smv'], trial['g_smv'])
    if pk is None:
        return None
    raw6 = np.concatenate([trial['accel'], trial['gyro']], axis=1)
    s = pk - FALL_PRE; e = pk + FALL_POST
    if s < 0 or e > len(raw6):
        return None
    return raw6[s:e].astype(np.float32)

def get_adl_windows(trial):
    raw6 = np.concatenate([trial['accel'], trial['gyro']], axis=1)
    a    = trial['a_smv']
    hard = []; easy = []
    for w in range(0, len(raw6) - WIN_SAMPLES + 1, STRIDE):
        pk  = float(a[w:w + WIN_SAMPLES].max())
        win = raw6[w:w + WIN_SAMPLES]
        if pk >= HARD_THRESH:
            hard.append((pk, win))
        elif pk < EASY_THRESH:
            easy.append(win)
    hard.sort(key=lambda x: -x[0])
    hw = [win for _, win in hard[:MAX_HARD]]
    if len(easy) > MAX_EASY:
        idx  = rng.choice(len(easy), MAX_EASY, replace=False)
        easy = [easy[i] for i in idx]
    return hw + easy

# ── Load CSV files ───────────────────────────────────────────────────
csv_files = sorted(glob.glob(os.path.join(CSV_DIR, '**', '*.csv'), recursive=True))
print(f'Found {len(csv_files)} CSV files in {CSV_DIR}')

fall_trials = []
adl_trials  = []
skipped     = []

for filepath in csv_files:
    fname  = os.path.basename(filepath)
    prefix = fname[0].upper()

    if prefix == 'F':
        label = 1
    elif prefix == 'D':
        label = 0
    else:
        skipped.append(fname)
        continue

    accel, gyro = load_csv(filepath)

    if accel is None:
        skipped.append(fname)
        print(f'  Warning: could not parse {fname}')
        continue

    trial = {
        'filename': fname,
        'filepath': filepath,
        'label':    label,
        'accel':    accel,
        'gyro':     gyro,
        'a_smv':    smv(accel),
        'g_smv':    smv(gyro),
    }

    if label == 1:
        fall_trials.append(trial)
    else:
        adl_trials.append(trial)

print(f'\nLoaded:')
print(f'  Fall trials : {len(fall_trials)}')
print(f'  ADL  trials : {len(adl_trials)}')
if skipped:
    print(f'  Skipped     : {skipped}')

# ── Build windows ────────────────────────────────────────────────────
windows = []; labels = []
fall_detected = 0; fall_skipped = 0
volunteer_stats = defaultdict(lambda: {'detected': [], 'skipped': []})

for trial in fall_trials:
    win       = get_fall_window(trial)
    volunteer = os.path.basename(os.path.dirname(trial['filepath']))
    if win is not None:
        windows.append(win); labels.append(1)
        fall_detected += 1
        volunteer_stats[volunteer]['detected'].append(trial['filename'])
    else:
        fall_skipped += 1
        volunteer_stats[volunteer]['skipped'].append(trial['filename'])

for trial in adl_trials:
    for win in get_adl_windows(trial):
        windows.append(win); labels.append(0)

X = np.array(windows, dtype=np.float32)
y = np.array(labels,  dtype=np.int32)

print('\n' + '=' * 55)
print('WINDOWING COMPLETE')
print('=' * 55)
print(f'Total windows  : {len(y):,}')
print(f'Fall windows   : {int(y.sum())}  (from {fall_detected}/{fall_detected+fall_skipped} trials detected)')
print(f'ADL windows    : {int((y==0).sum())}')
print(f'Fall %         : {100*y.mean():.1f}%')

# ── Per-volunteer breakdown ──────────────────────────────────────────
print(f'\n{"=" * 55}')
print('PER-VOLUNTEER FALL DETECTION BREAKDOWN')
print(f'{"=" * 55}')
print(f'{"Volunteer":<20} {"Detected":<10} {"Skipped":<10} {"Skipped Files"}')
print('-' * 55)
for vol in sorted(volunteer_stats.keys()):
    det      = len(volunteer_stats[vol]['detected'])
    skp      = len(volunteer_stats[vol]['skipped'])
    skp_files = ', '.join(volunteer_stats[vol]['skipped']) if skp > 0 else '—'
    print(f'{vol:<20} {det:<10} {skp:<10} {skp_files}')


Found 374 CSV files in /content/drive/MyDrive/M4/arduino_dataset/Data Collection

Loaded:
  Fall trials : 147
  ADL  trials : 222
  Skipped     : ['D11_01.csv', 'F04_05.csv', 'D07_03.csv', 'D17_01.csv', 'D02_01.csv']

WINDOWING COMPLETE
Total windows  : 1,698
Fall windows   : 111  (from 111/147 trials detected)
ADL windows    : 1587
Fall %         : 6.5%

PER-VOLUNTEER FALL DETECTION BREAKDOWN
Volunteer            Detected   Skipped    Skipped Files
-------------------------------------------------------
V10_Isham            13         10         F01_01.csv, F01_02.csv, F01_03.csv, F02_02.csv, F03_02.csv, F03_03.csv, F03_04.csv, F04_01.csv, F06_01.csv, F09_01.csv
V1_Sama              10         1          F01_01.csv
V2_Shams             23         4          F02_01.csv, F03_02.csv, F07_01.csv, F14_01.csv
V3_Mary              7          4          F02_01.csv, F02_02.csv, F03_01.csv, F03_02.csv
V4_Yasamin           9          2          F01_01.csv, F03_01.csv
V5_Betty             9      

In [ ]:
import matplotlib.pyplot as plt

# Collect all skipped fall trials
skipped_trials = [t for t in fall_trials if get_fall_window(t) is None]

print(f"Investigating {len(skipped_trials)} skipped fall trials...\n")

for trial in skipped_trials:
    a_smv = trial['a_smv']
    g_smv = trial['g_smv']
    pk    = int(np.argmax(a_smv))
    ap    = float(a_smv[pk])
    gs    = max(0, pk - 100); ge = min(len(g_smv), pk + 100)
    gp    = float(g_smv[gs:ge].max())

    vol = os.path.basename(os.path.dirname(trial['filepath']))

    # Check which condition failed
    ACC_THRESH = 1.5; GYR_THRESH = 90.0; HIGH_ACC = 4.0
    cond1 = ap >= ACC_THRESH
    cond2 = gp >= GYR_THRESH or ap >= HIGH_ACC

    # Check stillness
    MIN_STILL_local = MIN_STILL; STILL_MEAN_local = STILL_MEAN
    ss = -1
    for i in range(pk + 1, len(a_smv) - MIN_STILL_local):
        if a_smv[i:i + MIN_STILL_local].mean() < STILL_MEAN_local:
            ss = i; break
    cond3 = ss != -1

    # Check post-stillness
    if cond3:
        ch = min(ss + MIN_STILL_local, len(a_smv))
        cond4 = not (ch < len(a_smv) and float(a_smv[ch:].max()) > POST_STILL_MAX)
    else:
        cond4 = False

    print(f"[{vol}] {trial['filename']}")
    print(f"  Peak acc SMV : {ap:.3f}g  (need ≥ {ACC_THRESH})  → {'✅' if cond1 else '❌'}")
    print(f"  Peak gyro SMV: {gp:.1f} dps (need ≥ {GYR_THRESH}) → {'✅' if cond2 else '❌'}")
    print(f"  Stillness    : {'✅ found' if cond3 else '❌ not found'}")
    print(f"  Post-still   : {'✅' if cond4 else '❌ activity resumed'}")
    print()

Investigating 36 skipped fall trials...

[V10_Isham] F01_01.csv
  Peak acc SMV : 5.861g  (need ≥ 1.5)  → ✅
  Peak gyro SMV: 483.2 dps (need ≥ 90.0) → ✅
  Stillness    : ✅ found
  Post-still   : ❌ activity resumed

[V10_Isham] F01_02.csv
  Peak acc SMV : 8.411g  (need ≥ 1.5)  → ✅
  Peak gyro SMV: 538.9 dps (need ≥ 90.0) → ✅
  Stillness    : ✅ found
  Post-still   : ❌ activity resumed

[V10_Isham] F01_03.csv
  Peak acc SMV : 5.360g  (need ≥ 1.5)  → ✅
  Peak gyro SMV: 436.2 dps (need ≥ 90.0) → ✅
  Stillness    : ✅ found
  Post-still   : ❌ activity resumed

[V10_Isham] F02_02.csv
  Peak acc SMV : 2.564g  (need ≥ 1.5)  → ✅
  Peak gyro SMV: 233.4 dps (need ≥ 90.0) → ✅
  Stillness    : ✅ found
  Post-still   : ❌ activity resumed

[V10_Isham] F03_02.csv
  Peak acc SMV : 3.499g  (need ≥ 1.5)  → ✅
  Peak gyro SMV: 335.1 dps (need ≥ 90.0) → ✅
  Stillness    : ✅ found
  Post-still   : ❌ activity resumed

[V10_Isham] F03_03.csv
  Peak acc SMV : 3.769g  (need ≥ 1.5)  → ✅
  Peak gyro SMV: 557.8 dps (

In [ ]:
import os
import glob
import numpy as np
from collections import defaultdict

# ── Constants ────────────────────────────────────────────────────────
CSV_DIR     = "/content/drive/MyDrive/M4/arduino_dataset/Data Collection"
WIN_SAMPLES = 50
STRIDE      = 20
FALL_PRE    = 25
FALL_POST   = 25
MIN_STILL   = 50
STILL_MEAN  = 1.2
POST_STILL_MAX = 1.5
HARD_THRESH = 1.3
EASY_THRESH = 1.1
MAX_HARD    = 3
MAX_EASY    = 5
rng         = np.random.default_rng(42)

# ── Helper functions ─────────────────────────────────────────────────
def smv(a):
    return np.sqrt(np.sum(a ** 2, axis=1))

def load_csv(filepath):
    rows = []
    with open(filepath, "rb") as f:
        for i, line in enumerate(f):
            line = line.decode("latin-1").strip()
            if i == 0:
                if not line.startswith("t_ms"):
                    return None, None
                continue
            if not line or line.startswith("#") or "," not in line:
                continue
            parts = line.split(",")
            if len(parts) != 7:
                continue
            try:
                rows.append([float(v) for v in parts])
            except ValueError:
                continue
    if len(rows) < WIN_SAMPLES:
        return None, None
    arr   = np.array(rows, dtype=np.float32)
    accel = arr[:, 1:4]
    gyro  = arr[:, 4:7]
    return accel, gyro

def detect_impact(a_smv, g_smv):
    ACC_THRESH = 1.5
    GYR_THRESH = 90.0
    HIGH_ACC   = 4.0

    n  = len(a_smv)
    pk = int(np.argmax(a_smv))
    ap = float(a_smv[pk])

    # Condition 1+2: acc and gyro thresholds
    gs = max(0, pk - 100); ge = min(n, pk + 100)
    gp = float(g_smv[gs:ge].max())
    if not (ap >= ACC_THRESH and (gp >= GYR_THRESH or ap >= HIGH_ACC)):
        return None

    # Condition 3: post-impact stillness
    ss = -1
    for i in range(pk + 1, n - MIN_STILL):
        r = a_smv[i:i + MIN_STILL]
        if r.mean() < STILL_MEAN:
            ss = i; break
    if ss == -1:
        return None

    # Condition 4 REMOVED — volunteers naturally get up after falling

    return pk

def get_fall_window(trial):
    pk = detect_impact(trial['a_smv'], trial['g_smv'])
    if pk is None:
        return None
    raw6 = np.concatenate([trial['accel'], trial['gyro']], axis=1)
    s = pk - FALL_PRE; e = pk + FALL_POST
    if s < 0 or e > len(raw6):
        return None
    return raw6[s:e].astype(np.float32)

def get_adl_windows(trial):
    raw6 = np.concatenate([trial['accel'], trial['gyro']], axis=1)
    a    = trial['a_smv']
    hard = []; easy = []
    for w in range(0, len(raw6) - WIN_SAMPLES + 1, STRIDE):
        pk  = float(a[w:w + WIN_SAMPLES].max())
        win = raw6[w:w + WIN_SAMPLES]
        if pk >= HARD_THRESH:
            hard.append((pk, win))
        elif pk < EASY_THRESH:
            easy.append(win)
    hard.sort(key=lambda x: -x[0])
    hw = [win for _, win in hard[:MAX_HARD]]
    if len(easy) > MAX_EASY:
        idx  = rng.choice(len(easy), MAX_EASY, replace=False)
        easy = [easy[i] for i in idx]
    return hw + easy

# ── Load CSV files ───────────────────────────────────────────────────
csv_files = sorted(glob.glob(os.path.join(CSV_DIR, '**', '*.csv'), recursive=True))
print(f'Found {len(csv_files)} CSV files in {CSV_DIR}')

fall_trials = []
adl_trials  = []
skipped     = []

for filepath in csv_files:
    fname  = os.path.basename(filepath)
    prefix = fname[0].upper()

    if prefix == 'F':
        label = 1
    elif prefix == 'D':
        label = 0
    else:
        skipped.append(fname)
        continue

    accel, gyro = load_csv(filepath)

    if accel is None:
        skipped.append(fname)
        print(f'  Warning: could not parse {fname}')
        continue

    trial = {
        'filename': fname,
        'filepath': filepath,
        'label':    label,
        'accel':    accel,
        'gyro':     gyro,
        'a_smv':    smv(accel),
        'g_smv':    smv(gyro),
    }

    if label == 1:
        fall_trials.append(trial)
    else:
        adl_trials.append(trial)

print(f'\nLoaded:')
print(f'  Fall trials : {len(fall_trials)}')
print(f'  ADL  trials : {len(adl_trials)}')
if skipped:
    print(f'  Skipped     : {skipped}')

# ── Build windows ────────────────────────────────────────────────────
windows = []; labels = []
fall_detected = 0; fall_skipped = 0
volunteer_stats = defaultdict(lambda: {'detected': [], 'skipped': []})

for trial in fall_trials:
    win       = get_fall_window(trial)
    volunteer = os.path.basename(os.path.dirname(trial['filepath']))
    if win is not None:
        windows.append(win); labels.append(1)
        fall_detected += 1
        volunteer_stats[volunteer]['detected'].append(trial['filename'])
    else:
        fall_skipped += 1
        volunteer_stats[volunteer]['skipped'].append(trial['filename'])

for trial in adl_trials:
    for win in get_adl_windows(trial):
        windows.append(win); labels.append(0)

X = np.array(windows, dtype=np.float32)
y = np.array(labels,  dtype=np.int32)

print('\n' + '=' * 55)
print('WINDOWING COMPLETE')
print('=' * 55)
print(f'Total windows  : {len(y):,}')
print(f'Fall windows   : {int(y.sum())}  (from {fall_detected}/{fall_detected+fall_skipped} trials detected)')
print(f'ADL windows    : {int((y==0).sum())}')
print(f'Fall %         : {100*y.mean():.1f}%')

if fall_skipped > 0:
    print(f'\n⚠ {fall_skipped} fall trial(s) had no impact detected — discarded:')

# ── Per-volunteer breakdown ──────────────────────────────────────────
print(f'\n{"=" * 55}')
print('PER-VOLUNTEER FALL DETECTION BREAKDOWN')
print(f'{"=" * 55}')
print(f'{"Volunteer":<20} {"Detected":<10} {"Skipped":<10} {"Skipped Files"}')
print('-' * 55)
for vol in sorted(volunteer_stats.keys()):
    det       = len(volunteer_stats[vol]['detected'])
    skp       = len(volunteer_stats[vol]['skipped'])
    skp_files = ', '.join(volunteer_stats[vol]['skipped']) if skp > 0 else '—'
    print(f'{vol:<20} {det:<10} {skp:<10} {skp_files}')

Found 374 CSV files in /content/drive/MyDrive/M4/arduino_dataset/Data Collection

Loaded:
  Fall trials : 147
  ADL  trials : 222
  Skipped     : ['D11_01.csv', 'F04_05.csv', 'D07_03.csv', 'D17_01.csv', 'D02_01.csv']

WINDOWING COMPLETE
Total windows  : 1,730
Fall windows   : 143  (from 143/147 trials detected)
ADL windows    : 1587
Fall %         : 8.3%

⚠ 4 fall trial(s) had no impact detected — discarded:

PER-VOLUNTEER FALL DETECTION BREAKDOWN
Volunteer            Detected   Skipped    Skipped Files
-------------------------------------------------------
V10_Isham            23         0          —
V1_Sama              10         1          F01_01.csv
V2_Shams             25         2          F02_01.csv, F14_01.csv
V3_Mary              11         0          —
V4_Yasamin           11         0          —
V5_Betty             10         1          F11_01.csv
V6_Jannah            27         0          —
V7_Warda             26         0          —


## 5. Feature Extraction (20 Biomechanical Features for Gate)

In [ ]:
FEATURE_NAMES_20 = [
    'accel_x_max', 'accel_y_max', 'accel_z_max',
    'accel_x_std', 'accel_y_std', 'accel_z_std',
    'accel_x_peak2peak', 'accel_y_peak2peak', 'accel_z_peak2peak',
    'accel_y_mean_jerk',
    'accel_magnitude_sq_mean', 'gyro_magnitude_sq_mean',
    'accel_signal_magnitude_area', 'gyro_signal_magnitude_area',
    'accel_svm_rms', 'accel_3d_peak2peak', 'accel_3d_std',
    'accel_svm_max', 'accel_svm_std', 'gyro_magnitude_std'
]

def extract_20_features(window):
    wx  = window[:, 0].astype(np.float64)
    wy  = window[:, 1].astype(np.float64)
    wz  = window[:, 2].astype(np.float64)
    wxg = window[:, 3].astype(np.float64)
    wyg = window[:, 4].astype(np.float64)
    wzg = window[:, 5].astype(np.float64)

    features = []
    features.append(np.max(wx))
    features.append(np.max(wy))
    features.append(np.max(wz))
    features.append(np.std(wx))
    features.append(np.std(wy))
    features.append(np.std(wz))
    features.append(np.max(wx) - np.min(wx))
    features.append(np.max(wy) - np.min(wy))
    features.append(np.max(wz) - np.min(wz))
    features.append(np.mean(np.abs(np.diff(wy))))
    features.append(np.mean(wx**2 + wy**2 + wz**2))
    features.append(np.mean(wxg**2 + wyg**2 + wzg**2))
    features.append(np.mean(np.abs(wx) + np.abs(wy) + np.abs(wz)))
    features.append(np.mean(np.abs(wxg) + np.abs(wyg) + np.abs(wzg)))
    svm = np.sqrt(wx**2 + wy**2 + wz**2)
    features.append(np.sqrt(np.mean(svm**2)))
    features.append(np.sqrt((np.max(wx)-np.min(wx))**2 +
                             (np.max(wy)-np.min(wy))**2 +
                             (np.max(wz)-np.min(wz))**2))
    features.append(np.sqrt(np.std(wx)**2 + np.std(wy)**2 + np.std(wz)**2))
    gyro_mag = np.sqrt(wxg**2 + wyg**2 + wzg**2)
    features.append(np.max(svm))
    features.append(np.std(svm))
    features.append(np.std(gyro_mag))
    return np.array(features, dtype=np.float32)


print('Extracting 20 features...')
X_feat = np.array([extract_20_features(w) for w in X], dtype=np.float32)
X_feat = np.nan_to_num(X_feat, nan=0.0, posinf=0.0, neginf=0.0)
print(f'Done. Shape: {X.shape} → {X_feat.shape}')

Extracting 20 features...
Done. Shape: (1502, 50, 6) → (1502, 20)


## 6. Reconstruct Gate Model from Hardcoded Parameters

In [ ]:
SCALER_MEANS = np.array([
    0.42971454273754756, -0.2887337536218261, 0.33653926570932496,
    0.17680961572224266,  0.27045762159952963, 0.19510111703047683,
    0.8927497124657171,   1.2479464799168363,  0.9444061794988057,
    0.07834210888735628,  1.574355499439405,   6795.9364123993255,
    1.4019766041823933,  58.42353162687036,    1.1817498617464173,
    1.921234756391909,    0.4027520253407969,  2.120018342817862,
    0.31212664115634287, 22.0291616120648
], dtype=np.float64)

SCALER_STDS = np.array([
    1.0906016746100955, 1.1364981259988605, 0.9329609302951114,
    0.3588053383909056, 0.4427257569038268, 0.3272144758746891,
    1.9249997433888513, 2.1519916902413607, 1.7374088172676299,
    0.1442703293909912, 1.4480788079880127, 17613.298448316054,
    0.4145862102413671, 85.03158948616705,  0.4216903653073832,
    3.303091607734358,  0.6419352746416279, 2.094810274411141,
    0.5143170046182082, 34.58962470467294
], dtype=np.float64)

scaler = StandardScaler()
scaler.mean_  = SCALER_MEANS
scaler.scale_ = SCALER_STDS
scaler.var_   = SCALER_STDS ** 2
scaler.n_features_in_  = 20
scaler.n_samples_seen_ = 11303

RFE_FEATURE_NAMES   = ['gyro_signal_magnitude_area', 'accel_y_max',
                        'accel_y_mean_jerk', 'accel_svm_max', 'accel_z_std']
RFE_FEATURE_INDICES = [FEATURE_NAMES_20.index(f) for f in RFE_FEATURE_NAMES]

gate_model = LogisticRegression()
gate_model.classes_         = np.array([0, 1])
gate_model.intercept_       = np.array([-0.614884350201382])
gate_model.coef_            = np.array([[1.0181210950729982, 0.32774068140451756,
                                          0.0, 0.05188830974505934, 0.0]])
gate_model.n_features_in_   = 5

print('Gate model reconstructed.')
print(f'Active features : gyro_signal_magnitude_area, accel_y_max, accel_svm_max')
print(f'Threshold       : {GATE_THRESHOLD}')

Gate model reconstructed.
Active features : gyro_signal_magnitude_area, accel_y_max, accel_svm_max
Threshold       : 0.58


## 7. Load M3 CNN (TFLite)

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
CNN_INPUT_SHAPE = tuple(input_details[0]['shape'])

print(f'TFLite loaded. Input shape: {CNN_INPUT_SHAPE}  File: {os.path.getsize(TFLITE_PATH)/1024:.1f} KB')


def cnn_predict_batch(windows_raw):
    probs = []
    for w in windows_raw:
        inp = w.reshape(CNN_INPUT_SHAPE).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], inp)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details[0]['index'])
        probs.append(float(out.flatten()[0]))
    return np.array(probs)

TFLite loaded. Input shape: (np.int32(1), np.int32(50), np.int32(6))  File: 5.9 KB


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 8. Run Two-Stage Pipeline

In [ ]:
n_total = len(y)

# ── Gate ─────────────────────────────────────────────────────────────
X_scaled = scaler.transform(X_feat.astype(np.float64))
X_rfe    = X_scaled[:, RFE_FEATURE_INDICES]
gate_probs = gate_model.predict_proba(X_rfe)[:, 1]
gate_pass  = gate_probs >= GATE_THRESHOLD
gate_reject = ~gate_pass

n_pass   = int(np.sum(gate_pass))
n_reject = int(np.sum(gate_reject))

fall_passed   = int(np.sum(gate_pass   & (y == 1)))
fall_rejected = int(np.sum(gate_reject & (y == 1)))
adl_passed    = int(np.sum(gate_pass   & (y == 0)))
adl_rejected  = int(np.sum(gate_reject & (y == 0)))

print('=' * 55)
print('GATE RESULTS')
print('=' * 55)
print(f'Total windows    : {n_total:,}')
print(f'Passed to CNN    : {n_pass:,}  ({100*n_pass/n_total:.1f}%)')
print(f'Rejected (ADL)   : {n_reject:,}  ({100*n_reject/n_total:.1f}%)')
print()
n_falls = int(np.sum(y == 1))
n_adls  = int(np.sum(y == 0))
print(f'Among {n_falls} fall windows:')
print(f'  Passed to CNN  : {fall_passed}  (gate TPR = {fall_passed/max(1,n_falls):.4f})')
print(f'  Missed at gate : {fall_rejected}  ← not recoverable by CNN')
print()
print(f'Among {n_adls} ADL windows:')
print(f'  Correctly rejected : {adl_rejected}')
print(f'  Passed to CNN      : {adl_passed}  (false alarms)')

# ── CNN ──────────────────────────────────────────────────────────────
print(f'\nRunning CNN on {n_pass} windows...')
passed_indices = np.where(gate_pass)[0]
cnn_probs = cnn_predict_batch(X[passed_indices])
cnn_preds = (cnn_probs >= 0.5).astype(int)

# ── Combine ───────────────────────────────────────────────────────────
y_pred = np.zeros(n_total, dtype=int)
for i, idx in enumerate(passed_indices):
    y_pred[idx] = cnn_preds[i]

print('Pipeline complete.')

GATE RESULTS
Total windows    : 1,502
Passed to CNN    : 436  (29.0%)
Rejected (ADL)   : 1,066  (71.0%)

Among 140 fall windows:
  Passed to CNN  : 140  (gate TPR = 1.0000)
  Missed at gate : 0  ← not recoverable by CNN

Among 1362 ADL windows:
  Correctly rejected : 1066
  Passed to CNN      : 296  (false alarms)

Running CNN on 436 windows...
Pipeline complete.


## 9. Metrics

In [ ]:
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall      = sensitivity
accuracy    = (tp + tn) / (tp + tn + fp + fn)
f1          = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
macro_f1    = f1_score(y, y_pred, average='macro')
weighted_f1 = f1_score(y, y_pred, average='weighted')

print('=' * 55)
print('TWO-STAGE PIPELINE — COLLECTED DATA RESULTS')
print('=' * 55)
print(f'{"Metric":<30} {"Value":>10}')
print('-' * 42)
print(f'{"Overall accuracy":<30} {accuracy*100:>9.2f}%')
print(f'{"Sensitivity (fall recall)":<30} {sensitivity:>10.4f}')
print(f'{"Specificity (ADL recall)":<30} {specificity:>10.4f}')
print(f'{"Fall precision":<30} {precision:>10.4f}')
print(f'{"Fall F1-score":<30} {f1:>10.4f}')
print(f'{"Macro F1-score":<30} {macro_f1:>10.4f}')
print(f'{"Weighted F1-score":<30} {weighted_f1:>10.4f}')
print('-' * 42)
print(f'{"Total windows":<30} {n_total:>10,}')
print(f'{"Fall windows":<30} {n_falls:>10,}')
print(f'{"ADL windows":<30} {n_adls:>10,}')
print('=' * 55)
print(f'Confusion matrix:')
print(f'  TP={tp}  FN={fn}')
print(f'  FP={fp}  TN={tn}')
print()
print(f'Gate efficiency:')
print(f'  CNN executions saved : {100*n_reject/n_total:.1f}%  ({n_reject}/{n_total} windows)')

TWO-STAGE PIPELINE — COLLECTED DATA RESULTS
Metric                              Value
------------------------------------------
Overall accuracy                   79.83%
Sensitivity (fall recall)          0.9429
Specificity (ADL recall)           0.7834
Fall precision                     0.3091
Fall F1-score                      0.4656
Macro F1-score                     0.6706
Weighted F1-score                  0.8374
------------------------------------------
Total windows                       1,502
Fall windows                          140
ADL windows                         1,362
Confusion matrix:
  TP=132  FN=8
  FP=295  TN=1067

Gate efficiency:
  CNN executions saved : 71.0%  (1066/1502 windows)


## 10. Full Results Table (for Report)

In [ ]:
# ── Change CONDITION below to 'controlled' or 'new_condition' ─────────
CONDITION = 'controlled'   # or 'new_condition'
# ─────────────────────────────────────────────────────────────────────

summary = pd.DataFrame({
    'Evaluation Setting': [
        'M2 baseline CNN (5 KB) — Offline SisFall',
        'M3 two-stage Gate+CNN (64 KB) — Offline SisFall',
        'M2 on-device (hardware current only)',
        'M3 two-stage on-device (hardware current only)',
        f'Live {CONDITION} (fine-tuned)',
        'Live new condition (robustness test)' if CONDITION == 'controlled' else 'Live controlled (fine-tuned)',
    ],
    'Sensitivity (%)': ['97.40', '99.63', 'N/A', 'N/A',
                         f'{sensitivity*100:.2f}', '—'],
    'Specificity (%)': ['99.41', '99.17', 'N/A', 'N/A',
                         f'{specificity*100:.2f}', '—'],
    'Precision (%)':   ['95.62', '94.04', 'N/A', 'N/A',
                         f'{precision*100:.2f}', '—'],
    'Recall (%)':      ['97.40', '99.63', 'N/A', 'N/A',
                         f'{recall*100:.2f}', '—'],
    'Accuracy (%)':    ['99.18', '99.22', 'N/A', 'N/A',
                         f'{accuracy*100:.2f}', '—'],
    'Current (mA)': [
        '21.0 (CNN alone)',
        '—',
        'Idle: 18.8 / CNN: 21.0',
        'Idle: 18.8 / Gate: 19.1 / CNN: 20.9 / G+C: 21.1',
        '—', '—'
    ],
    'Latency (ms)': ['—', '—', '—', '—', '—', '—']
})

print(summary.to_string(index=False))
out_csv = f'evaluation_results_{CONDITION}.csv'
summary.to_csv(out_csv, index=False)
print(f'\nSaved: {out_csv}')

                             Evaluation Setting Sensitivity (%) Specificity (%) Precision (%) Recall (%) Accuracy (%)                                    Current (mA) Latency (ms)
       M2 baseline CNN (5 KB) — Offline SisFall           97.40           99.41         95.62      97.40        99.18                                21.0 (CNN alone)            —
M3 two-stage Gate+CNN (64 KB) — Offline SisFall           99.63           99.17         94.04      99.63        99.22                                               —            —
           M2 on-device (hardware current only)             N/A             N/A           N/A        N/A          N/A                          Idle: 18.8 / CNN: 21.0            —
 M3 two-stage on-device (hardware current only)             N/A             N/A           N/A        N/A          N/A Idle: 18.8 / Gate: 19.1 / CNN: 20.9 / G+C: 21.1            —
                   Live controlled (fine-tuned)           94.29           78.34         30.91      94.29 